In [ ]:
import math
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from utils import load_video, play_video

In [ ]:
video = load_video("data/calib.h264", gray=True)
print(video.shape)

In [ ]:
play_video(video[::10])

In [ ]:
plt.figure(figsize=(12, 8))
plt.imshow(video[250], cmap="gray")

In [ ]:
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
height, width = video.shape[1:3]
pattern_size = (5, 3)
square_size = 0.070
objp = np.zeros((pattern_size[0] * pattern_size[1], 3), np.float32)
objp[:, :2] = (
    np.mgrid[0 : pattern_size[0], 0 : pattern_size[1]].T.reshape(-1, 2) * square_size
)

In [ ]:
objp

In [ ]:
obj_points = []
img_points = []
for i in tqdm(range(0, video.shape[0], 10)):
    frame = video[i]
    ret, corners = cv2.findChessboardCorners(frame, pattern_size, None)
    if ret:
        obj_points.append(objp)
        refined_corners = cv2.cornerSubPix(frame, corners, (11, 11), (-1, -1), criteria)
        img_points.append(refined_corners)
print(f"Found {len(obj_points)} valid frames")

In [ ]:
# num_trials = 200  # how many random calibrations
# sample_size = 20  # number of points per sample

# mtx_list = []

# for _ in range(num_trials):
#     keep_idx = np.random.choice(len(obj_points), size=sample_size, replace=False)
#     kept_obj = [obj_points[i] for i in keep_idx]
#     kept_img = [img_points[i] for i in keep_idx]

#     ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(
#         kept_obj, kept_img, (width, height), None, None
#     )
#     mtx_list.append(mtx)

# mtx_arr = np.array(mtx_list)  # shape: (num_trials, 3, 3)

# fig, axes = plt.subplots(3, 3, figsize=(10, 10))

# for r in range(3):
#     for c in range(3):
#         axes[r, c].hist(mtx_arr[:, r, c], bins=20)
#         axes[r, c].set_title(f"mtx[{r},{c}]")

# plt.tight_layout()
# plt.show()

In [ ]:
# keep_indices = np.random.choice(len(obj_points), size=len(obj_points), replace=False)
# kept_obj_points = [obj_points[i] for i in keep_indices]
# kept_img_points = [img_points[i] for i in keep_indices]

ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(
    obj_points, img_points, (width, height), None, None
)
new_mtx, roi = cv2.getOptimalNewCameraMatrix(
    mtx, dist, (width, height), 0, (width, height)
)
# np.savez("calib_data.npz", mtx=mtx, dist=dist, new_mtx=new_mtx, roi=roi)
print(mtx)
print(dist)
print(new_mtx)
print(roi)

In [ ]:
np.savez("data/calib/calib_data1.npz", mtx=mtx, dist=dist, new_mtx=new_mtx, roi=roi)

In [ ]:
new_mtx, roi = cv2.getOptimalNewCameraMatrix(
    mtx, dist, (width, height), 0, (width, height)
)
# np.savez("calib_data.npz", mtx=mtx, dist=dist, new_mtx=new_mtx, roi=roi)
print(mtx)
print(dist)
print(new_mtx)
print(roi)

In [ ]:
# np.savez("data/calib/calib_data.npz", mtx=mtx, dist=dist, new_mtx=new_mtx, roi=roi)

In [ ]:
data = np.load("data/calib/calib_data.npz")
mtx = data["mtx"]
dist = data["dist"]
new_mtx = data["new_mtx"]
roi = data["roi"]

In [ ]:
mtx

In [ ]:
frame = video[250]
undistorted = cv2.undistort(frame, mtx, dist, None, new_mtx)
# show undistorted image and original side by side
plt.figure(figsize=(16, 8))
plt.subplot(1, 2, 1)
plt.title("Original")
plt.imshow(frame, cmap="gray")
plt.subplot(1, 2, 2)
plt.title("Undistorted")
plt.imshow(undistorted, cmap="gray")

In [ ]:
mean_error = 0
for i in range(len(obj_points)):
    img_points2, _ = cv2.projectPoints(obj_points[i], rvecs[i], tvecs[i], mtx, dist)
    error = cv2.norm(img_points[i], img_points2, cv2.NORM_L2) / len(img_points2)
    mean_error += error

print("total error: {}".format(mean_error / len(obj_points)))

In [ ]:
roi